# Session 1: FHIR Fundamentals

## How This Notebook Works

In this session you will **query a real FHIR server** containing synthetic (fake but clinically realistic) patient data. You will:

1. Explore the FHIR API structure (Bundles, Resources, References)
2. Use the **Claude web UI** (claude.ai) to help you write Python code for FHIR queries
3. Find patients with Type 2 diabetes
4. Retrieve their HbA1c, creatinine, and eGFR lab values
5. Identify patients with both **poor glycemic control** and **impaired kidney function**
6. Visualize the relationship between diabetes control and kidney function

**You write the code** (with Claude's help), paste it into the empty cells, and run it. Each step builds on the previous one.

## About the Patient Population

This FHIR server contains **1,027 synthetic patients** generated with clinically coupled phenotypes. The patients are distributed across **6 clinical groups**:

| # | Phenotype | Description |
|---|-----------|-------------|
| 1 | **Metabolic syndrome** | Elevated BMI, blood pressure, triglycerides, and fasting glucose — but no diabetes diagnosis yet. |
| 2 | **Early Type 2 diabetes** | Recently diagnosed Type 2 diabetes with mildly elevated HbA1c. Kidney function is normal. |
| 3 | **Type 2 diabetes with chronic kidney disease stage G2** | Established Type 2 diabetes with mildly reduced kidney function (eGFR 60–89). |
| 4 | **Advanced Type 2 diabetes with chronic kidney disease stage G3b** | Long-standing Type 2 diabetes with moderately-to-severely reduced kidney function (eGFR 30–44). Often on insulin and multiple medications. |
| 5 | **Type 1 diabetes with early nephropathy** | Type 1 diabetes (the body's immune system destroys insulin-producing cells) with early signs of kidney damage. Very low C-peptide. |
| 6 | **Type 1 diabetes with poor control and chronic kidney disease stage G3a** | Type 1 diabetes with poor glycemic control (high HbA1c) and moderately reduced kidney function (eGFR 45–59). |

These phenotypes are **clinically coupled** — patients with worse diabetes control tend to have worse kidney function, mirroring real-world patterns.

## Clinical Code Reference

### Diagnosis Codes (SNOMED CT)

| Code | Condition | Notes |
|------|-----------|-------|
| 44054006 | Type 2 diabetes mellitus | The body becomes resistant to insulin |
| 46635009 | Type 1 diabetes mellitus | The immune system destroys insulin-producing cells |
| 709044004 | Chronic kidney disease | Gradual loss of kidney function over time |

### Observation Codes (LOINC)

| Code | Test | What It Measures |
|------|------|-----------------|
| 4548-4 | Hemoglobin A1c (HbA1c) | Average blood sugar over 2–3 months |
| 2160-0 | Creatinine | Waste product filtered by kidneys |
| 33914-3 | Estimated glomerular filtration rate (eGFR) | How well kidneys filter blood |
| 14959-1 | Urine albumin/creatinine ratio (UACR) | Protein leakage indicating kidney damage |
| 1986-9 | C-peptide | Marker of insulin production by the pancreas |
| 85354-9 | Blood pressure panel | Systolic and diastolic blood pressure |
| 39156-5 | Body mass index (BMI) | Weight relative to height |
| 1558-6 | Fasting glucose | Blood sugar after overnight fasting |
| 2339-0 | Blood glucose | Random blood sugar measurement |
| 3094-0 | Blood urea nitrogen (BUN) | Another kidney function marker |
| 13457-7 | LDL cholesterol | "Bad" cholesterol |
| 2085-9 | HDL cholesterol | "Good" cholesterol |
| 2571-8 | Triglycerides | Blood fat linked to heart disease risk |

### Interpretation Thresholds

| Measure | Range | Interpretation |
|---------|-------|----------------|
| HbA1c | < 5.7% | Normal |
| | 5.7% – 6.4% | Prediabetes |
| | ≥ 6.5% | Diabetes |
| | **> 7.5%** | **Poor glycemic control** |
| eGFR | > 90 | Normal kidney function |
| | 60–89 | Mildly decreased |
| | 45–59 | Moderately decreased |
| | 30–44 | Moderately-to-severely decreased |
| | < 30 | Severely decreased |
| UACR | < 30 mg/g | Normal |
| | 30–300 mg/g | Moderately increased albuminuria |
| | > 300 mg/g | Severely increased albuminuria |

## How FHIR Resources Link Together

```
                    ┌─────────────┐
                    │   Patient   │
                    │  (who)      │
                    └──────┬──────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
       ┌──────▼──────┐ ┌──▼──────────┐ ┌▼────────────────┐
       │  Condition   │ │ Observation │ │MedicationRequest│
       │  (diagnosis) │ │ (lab/vital) │ │ (prescription)  │
       └─────────────┘ └─────────────┘ └─────────────────┘

Each resource has a "subject" reference pointing back to the Patient.
To answer clinical questions, you follow these references to connect
diagnoses → patients → lab results → medications.
```

In [ ]:
!pip install requests pandas matplotlib

import requests
import urllib3
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Suppress SSL warnings (self-signed cert on teaching server)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---- FHIR Server Configuration ----
FHIR_BASE = "https://lfh-fhir.eastus2.cloudapp.azure.com:9443/fhir-server/api/v4"

# Credentials (hardcoded — this is synthetic teaching data only)
FHIR_SESSION = requests.Session()
FHIR_SESSION.auth = ("fhiruser", "BmI512@ccess")
FHIR_SESSION.verify = False

# Helper for exploring JSON responses
def show_json(data, max_lines=30):
    """Pretty-print JSON with optional truncation."""
    text = json.dumps(data, indent=2)
    lines = text.split("\n")
    if len(lines) > max_lines:
        print("\n".join(lines[:max_lines]))
        print(f"\n... ({len(lines) - max_lines} more lines)")
    else:
        print(text)

# Verify connection
resp = FHIR_SESSION.get(f"{FHIR_BASE}/metadata", params={"_format": "json"}, timeout=10)
if resp.status_code == 200:
    fhir_version = resp.json().get("fhirVersion", "unknown")
    print(f"\u2705 Connected to FHIR server (version {fhir_version})")
    count_resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Patient",
        params={"_summary": "count", "_format": "json"},
        timeout=10
    )
    if count_resp.status_code == 200:
        total = count_resp.json().get("total", "unknown")
        print(f"   {total} patients available")
    print(f"   Authenticated as: fhiruser")
else:
    print(f"\u274c Could not connect: HTTP {resp.status_code}")

## Step 0: Your First FHIR Query

Let's start by fetching a few patients to see what FHIR data looks like. The cell below queries `/Patient?_count=5` — asking the server for 5 patient records.

Look at the response structure: it's a **Bundle** containing **entry** items, each wrapping a **Patient** resource.

In [ ]:
# Your first FHIR query: fetch 5 patients
resp = FHIR_SESSION.get(
    f"{FHIR_BASE}/Patient",
    params={"_count": 5, "_format": "json"},
    timeout=30,
)
bundle = resp.json()

print(f"Response type: {bundle['resourceType']}")
print(f"Total patients on server: {bundle.get('total', '?')}")
print(f"Entries returned: {len(bundle.get('entry', []))}")
print()

# Show first patient
if bundle.get("entry"):
    first = bundle["entry"][0]["resource"]
    show_json(first)

## What Did We Just See?

The FHIR server returned a **Bundle** — a container wrapping multiple results. Key fields:

- `resourceType`: always `"Bundle"` for search results
- `total`: how many resources matched on the entire server
- `entry`: an array of results, each containing one `resource`

Each Patient resource has structured fields: `name`, `gender`, `birthDate`, and an `id` used to reference this patient from other resources.

Now let's use this structure to answer a clinical question.

## Step 1: Find Patients with Type 2 Diabetes

Go to **claude.ai** and ask Claude to write code for you. Use a prompt like:

> Write Python code using `FHIR_SESSION.get()` (a pre-configured requests Session with auth) to search for Condition resources with SNOMED CT code `44054006` (Type 2 diabetes mellitus) on the FHIR server. The base URL is stored in `FHIR_BASE`. Use `_count=50` and `_format=json`. Extract the unique patient references into a set called `patient_refs`.

Paste Claude's code into the cell below and run it.

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ---- Verification ----
try:
    assert len(patient_refs) > 0, "No patient references found"
    print(f"\u2705 Found {len(patient_refs)} patient references")
    for ref in sorted(patient_refs)[:5]:
        print(f"   {ref}")
    if len(patient_refs) > 5:
        print(f"   ... and {len(patient_refs) - 5} more")
except (NameError, AssertionError) as e:
    print(f"\u26a0\ufe0f Issue: {e}")
    print("Running fallback query...")
    resp = FHIR_SESSION.get(
        f"{FHIR_BASE}/Condition",
        params={"code": "44054006", "_count": 50, "_format": "json"},
        timeout=30,
    )
    bundle = resp.json()
    patient_refs = set()
    for entry in bundle.get("entry", []):
        ref = entry["resource"]["subject"]["reference"]
        patient_refs.add(ref)
    print(f"\u2705 Fallback found {len(patient_refs)} patient references")

## Step 2: Get Patient Demographics

Now ask Claude to write code that:

> For each patient reference in `patient_refs`, fetch the Patient resource from `f"{FHIR_BASE}/Patient/{patient_id}"` using `FHIR_SESSION.get()`. Extract the patient's name, gender, and birth date into a list of dictionaries called `patients`. Display the results as a pandas DataFrame.

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ---- Verification: Patient demographics ----
try:
    df_patients = pd.DataFrame(patients)
    print(f"\u2705 {len(patients)} patients loaded")
    display(df_patients)
except NameError:
    print("\u26a0\ufe0f `patients` not defined. Run Step 2 first.")

## Step 3: Retrieve Lab Values (HbA1c, Creatinine, eGFR)

This is the most complex step. Ask Claude:

> For each patient in `patient_refs`, search for Observation resources using `FHIR_SESSION.get()` at `f"{FHIR_BASE}/Observation"`. Retrieve three lab types:
> - HbA1c (LOINC code `4548-4`)
> - Creatinine (LOINC code `2160-0`)
> - eGFR (LOINC code `33914-3`)
>
> Use parameters: `subject=Patient/{id}`, `code={loinc}`, `_count=1`, `_sort=-date`, `_format=json`. Extract the most recent value for each test into a list of dictionaries called `observations` with keys: patient_id, test, loinc_code, value, unit, date.

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ══════════════════════════════════════════════════════════════
# COMBINED ANALYSIS
# ══════════════════════════════════════════════════════════════

df_patients = pd.DataFrame(patients)
df_obs = pd.DataFrame(observations)

# Pivot: one row per patient, columns for each test
df_pivot = df_obs.pivot(index="patient_id", columns="test", values="value").reset_index()
df_combined = df_patients.merge(df_pivot, left_on="id", right_on="patient_id", how="left")

# Convert to numeric
for col in ["HbA1c", "Creatinine", "eGFR"]:
    if col in df_combined.columns:
        df_combined[col] = pd.to_numeric(df_combined[col], errors="coerce")

# Flag poor control and impaired kidney function
df_combined["poor_control"] = df_combined["HbA1c"] > 7.5
df_combined["impaired_kidneys"] = (
    (df_combined["eGFR"] < 60) | (df_combined["Creatinine"] > 1.5)
)
df_combined["both_flags"] = df_combined["poor_control"] & df_combined["impaired_kidneys"]

print("=" * 70)
print("COMBINED ANALYSIS: Type 2 Diabetes Patients")
print("=" * 70)
display(df_combined[["name", "gender", "birthDate", "HbA1c", "Creatinine", "eGFR",
                     "poor_control", "impaired_kidneys"]])

print(f"\n--- Summary ---")
print(f"Total patients analyzed: {len(df_combined)}")
n_hba1c = df_combined["HbA1c"].notna().sum()
print(f"Patients with HbA1c data: {n_hba1c}")
if n_hba1c > 0:
    print(f"  Poor glycemic control (HbA1c >7.5%): {df_combined['poor_control'].sum()}")
    print(f"  Mean HbA1c: {df_combined['HbA1c'].mean():.1f}%")
n_egfr = df_combined["eGFR"].notna().sum()
print(f"Patients with eGFR data: {n_egfr}")
if n_egfr > 0:
    print(f"  Impaired kidneys (eGFR <60 or Cr >1.5): {df_combined['impaired_kidneys'].sum()}")
    print(f"  Mean eGFR: {df_combined['eGFR'].mean():.1f} mL/min/1.73m\u00b2")
print(f"\nPatients with BOTH poor control AND impaired kidneys: {df_combined['both_flags'].sum()}")

In [ ]:

# ══════════════════════════════════════════════════════════════
# VISUALIZATION: HbA1c vs eGFR
# ══════════════════════════════════════════════════════════════

# Pivot observations to get one row per patient with columns for each test
df_pivot = df_obs.pivot(index="patient_id", columns="test", values="value").reset_index()
df_plot = df_patients.merge(df_pivot, left_on="id", right_on="patient_id", how="left")

# Convert to numeric
for col in ["HbA1c", "Creatinine", "eGFR"]:
    if col in df_plot.columns:
        df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce")

# Filter to patients with both values
plot_data = df_plot.dropna(subset=["HbA1c", "eGFR"])

if len(plot_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ["tab:red" if h > 7.5 else "tab:green" for h in plot_data["HbA1c"]]
    ax.scatter(plot_data["HbA1c"], plot_data["eGFR"], c=colors, alpha=0.7,
               edgecolors="black", linewidth=0.5, s=80)
    ax.axhline(y=60, color="gray", linestyle="--", alpha=0.7, label="eGFR = 60 (impaired)")
    ax.axvline(x=7.5, color="gray", linestyle=":", alpha=0.7, label="HbA1c = 7.5% (poor control)")
    ax.set_xlabel("HbA1c (%)", fontsize=12)
    ax.set_ylabel("eGFR (mL/min/1.73m\u00b2)", fontsize=12)
    ax.set_title("Glycemic Control vs Kidney Function in Type 2 Diabetes Patients", fontsize=13)
    ax.legend(fontsize=10)

    # Label quadrants
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    mid_x = 7.5
    ax.text(xlim[0] + 0.3, ylim[1] - 5, "Good control,\nhealthy kidneys",
            fontsize=9, color="green", alpha=0.8, va="top")
    ax.text(xlim[1] - 0.3, ylim[1] - 5, "Poor control,\nhealthy kidneys",
            fontsize=9, color="orange", alpha=0.8, va="top", ha="right")
    ax.text(xlim[0] + 0.3, max(ylim[0] + 2, 10), "Good control,\nimpaired kidneys",
            fontsize=9, color="orange", alpha=0.8)
    ax.text(xlim[1] - 0.3, max(ylim[0] + 2, 10), "Poor control,\nimpaired kidneys",
            fontsize=9, color="red", alpha=0.8, ha="right")

    plt.tight_layout()
    plt.show()

    print(f"\nPatients plotted: {len(plot_data)}")
    print(f"Poor control (HbA1c >7.5%): {sum(plot_data['HbA1c'] > 7.5)}")
    print(f"Impaired kidneys (eGFR <60): {sum(plot_data['eGFR'] < 60)}")
    both = sum((plot_data['HbA1c'] > 7.5) & (plot_data['eGFR'] < 60))
    print(f"Both poor control AND impaired kidneys: {both}")
else:
    print("Not enough data to plot. Check that observations were retrieved successfully.")


## Step 4: Generate a Clinical Summary

Go back to **claude.ai** and ask Claude to write a clinical narrative based on the data table above. Suggested prompt:

> Based on the following patient data, write a brief clinical summary describing the relationship between glycemic control (HbA1c) and kidney function (eGFR, creatinine) in these Type 2 diabetes patients. Use ONLY the data provided. Identify which patients have both poor glycemic control and impaired kidney function.

Paste the summary below.

### Your Clinical Summary

*Paste Claude's clinical summary here.*


## Session 1 Reflection

You just built a complete clinical data pipeline:

1. **Searched** for a diagnosis (Condition) by SNOMED code
2. **Followed references** to get Patient demographics
3. **Retrieved** lab values (Observations) by LOINC codes
4. **Analyzed** the combined data to flag at-risk patients
5. **Visualized** the relationship between glycemic control and kidney function

**Key insight:** No single FHIR query answered our clinical question. We had to chain multiple queries together by following references. In Session 2, you'll see how an AI agent can decide this sequence automatically.